# Day 14 Exercises: Revision & Debugging

In this notebook, we'll practice:
1. Identifying and fixing a **SettingWithCopyWarning**.
2. Profiling and optimizing a slow **row-wise `.apply(axis=1)`** into a vectorized operation.

In [1]:
import numpy as np
import pandas as pd
import time

# Create a mock dataset
np.random.seed(42)
n_rows = 100000

data = {
    'OrderID': range(1, n_rows + 1),
    'Sales': np.random.uniform(10, 2000, n_rows),
    'Quantity': np.random.randint(1, 10, n_rows),
    'Category': np.random.choice(['Furniture', 'Office Supplies', 'Technology'], n_rows),
    'Segment': np.random.choice(['Consumer', 'Corporate', 'Home Office'], n_rows)
}

df = pd.DataFrame(data)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (100000, 5)


,OrderID,Sales,Quantity,Category,Segment
0,1,755.334837,2,Furniture,Home Office
1,2,1901.921470,8,Technology,Consumer
2,3,1466.667944,9,Technology,Consumer
3,4,1201.330384,1,Technology,Home Office
4,5,320.477094,6,Technology,Consumer


## Exercise 1: SettingWithCopyWarning

**Goal:** Correctly categorize orders with Sales > 1500 as "Premium" in the `Segment` column.

### Step-by-Step Explanation:
1. **The Warning:** When you slice a DataFrame (e.g., `df_subset = df[df['Sales'] > 1500]`), Pandas returns a *view* of the original DataFrame. When you attempt to assign values to this view (e.g., `df_subset['Segment'] = 'Premium'`), Pandas triggers a `SettingWithCopyWarning` because it cannot guarantee whether your modifications will update the original DataFrame or just the slice.
2. **Fix A (In-place using `.loc`):** If you want to modify the original DataFrame, access it directly using `df.loc[row_selector, column_selector] = value`. This bypasses temporary views and updates `df` directly.
3. **Fix B (Explicit Copy using `.copy()`):** If you want to create a standalone subset DataFrame that you can modify without affecting the original, explicitly use `.copy()` to clone it in memory.

In [2]:
# WARNING: This will trigger the SettingWithCopyWarning!
df_subset = df[df['Sales'] > 1500]
df_subset['Segment'] = 'Premium'

### Fix A: Modify the original DataFrame in-place using `.loc`
*(Hint: Access and modify `df` directly using row/column selection in `.loc`)*

In [3]:
# Reset the data first
df = pd.DataFrame(data)

# Write your Fix A code here:
df.loc[df['Sales'] > 1500, 'Segment'] = 'Premium'

# Verify it updated the original dataframe:
print("Premium count in original df:", (df['Segment'] == 'Premium').sum())

Premium count in original df: 25093


### Fix B: Create a standalone copy of the slice using `.copy()`
*(Hint: Create a true copy of the sliced dataframe so that modification warnings are avoided and the original DataFrame remains unaffected)*

In [4]:
# Reset the data first
df = pd.DataFrame(data)

# Write Fix B here:
df_subset_copy = df[df['Sales'] > 1500].copy()
df_subset_copy['Segment'] = 'Premium'

# Verify that df_subset_copy has updated and df remained unchanged:
print("Premium count in copy:", (df_subset_copy['Segment'] == 'Premium').sum())
print("Premium count in original df:", (df['Segment'] == 'Premium').sum())

Premium count in copy: 25093
Premium count in original df: 0


## Exercise 2: Vectorizing a Slow `.apply()`

**Goal:** Calculate shipping costs based on category and sales:
- **Technology**: Shipping is `Sales * 0.02`
- **Furniture**: Shipping is `Sales * 0.05`
- **Office Supplies**: Shipping is `Sales * 0.01`

### Step-by-Step Explanation:
1. **The Row-wise `.apply()` Method:** Passing `axis=1` to `.apply()` forces Pandas to process rows sequentially like a Python `for` loop. It creates a `Series` object for each of the 100,000 rows and executes the Python function `calculate_shipping` on it, introducing a massive overhead.
2. **Vectorization:** Instead of looping row-by-row, vectorized operations perform calculations on entire column arrays at the compiled C-level via NumPy. 
3. **Using `np.select`:** For conditional logic in vectorization, `np.select` takes a list of boolean condition masks and a list of choice evaluations, assigning them in parallel. A default choice is specified for any rows that do not match the conditions (our Office Supplies baseline).

In [5]:
def calculate_shipping(row):
    if row['Category'] == 'Technology':
        return row['Sales'] * 0.02
    elif row['Category'] == 'Furniture':
        return row['Sales'] * 0.05
    else:
        return row['Sales'] * 0.01

# Benchmark the slow apply method
start_time = time.time()
df['Shipping_Apply'] = df.apply(calculate_shipping, axis=1)
apply_time = time.time() - start_time
print(f"Time taken by .apply(): {apply_time:.4f} seconds")

Time taken by .apply(): 0.5848 seconds


### Task: Vectorize the shipping logic using `numpy.select` (or `numpy.where`)
*(Avoid using any Python loops or `.apply()`)*

In [6]:
# Write the vectorized code here:
start_time = time.time()

# Define the conditions (logical criteria)
conditions = [
    df['Category'] == 'Technology',
    df['Category'] == 'Furniture'
]

# Define the outputs corresponding to each condition
choices = [
    df['Sales'] * 0.02,
    df['Sales'] * 0.05
]

# Run the vectorized selection (using Office Supplies' rate as the default)
df['Shipping_Vectorized'] = np.select(conditions, choices, default=df['Sales'] * 0.01)

vectorized_time = time.time() - start_time
print(f"Time taken by vectorized approach: {vectorized_time:.4f} seconds")
print(f"Speedup: {apply_time / vectorized_time:.1f}x")

# Check if the results are identical
print("Results match:", np.allclose(df['Shipping_Apply'], df['Shipping_Vectorized']))

Time taken by vectorized approach: 0.0051 seconds
Speedup: 115.0x
Results match: True
